In [4]:
import numpy as np
import pandas as pd
import os

In [20]:
name = 'hori_geomasked_timecut'
path = f'/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/{name}.bin'
num_cols = 4
data_type = np.int16

num_rows = os.path.getsize(path) // (num_cols * data_type().itemsize)
data = np.memmap(path, dtype = data_type, mode = 'r', shape = (num_rows, num_cols))
data = data[:,[0,1,2]]
print(num_rows)
print(data[0:5])
print(data[-6:-1])

12601
[[1918 1401  325]
 [1805 1771 -253]
 [1165 1394 -655]
 [1800 1389 -772]
 [1802 1305  133]]
[[1858 1674  262]
 [1191 1405 -440]
 [1243 1773 -457]
 [1245 1774 -831]
 [1158 1392 -845]]


In [21]:
# Remapping
def remap(idx):

    if 1152 <= idx <= 1279:
        return idx - 1152

    elif 1792 <= idx <= 1919:
        return (idx - 1792) + 128

    elif 1280 <= idx <= 1407:
        return (idx - 1280)

    elif 1664 <= idx <= 1791:
        return (idx - 1664) + 128

    else:
        return None

remapped_PET = []

for row in data:
    IDL, IDR, t = row

    new_IDL = remap(IDL)
    new_IDR = remap(IDR)

    if new_IDL is not None and new_IDR is not None:
        remapped_PET.append([new_IDL, new_IDR, t])

remapped_PET = np.array(remapped_PET, dtype=np.int16)

remapped_PET.tofile(f'/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/{name}_remap.bin')

print(remapped_PET[0:5])

[[ 254  121  325]
 [ 141  235 -253]
 [  13  114 -655]
 [ 136  109 -772]
 [ 138   25  133]]


In [7]:
# Map remapping
mapdf = pd.read_csv('/home/kale-chen/Documents/PET/TPPT_Scanner_map_adjusted.csv', usecols=[0,1,2,3,4,5], header=None)
map = mapdf.to_numpy()

section1 = map[1152:1280]
section2 = map[1792:1920]
section3 = map[4352:4480]
section4 = map[4736:4864]

total = np.concatenate((section1, section2, section3, section4), axis=0)

pd.DataFrame(total).to_csv('/home/kale-chen/Documents/PET/remap.csv', index=False, header=False)


